In [ ]:
# Parameters
bronze_schema = "bronze"
dbx_sql_source = "lh_databricks.dbo"    # SQL endpoint used to read the catalog-managed Databricks tables

# 02 - Bronze: validate the zero-copy mirrored sources

Mirrored data is **not copied**. `bronze.dbx_*` and `bronze.ref_* / erp_* / wms_*` in `lh_medallion` are OneLake
shortcuts (created by notebook `00_lh_medallion_setup`) to the mirrored items:

* `dbx_<table>` -> mirrored Azure Databricks catalog `mc_appareliq_uc` (Unity Catalog `appareliq.kinetiq_batch`, DS-01, DS-02)
* `<schema>_<table>` -> mirrored Azure SQL database `md_erp_wms` (ERP/WMS, DS-03..10, DS-19)

This notebook only reads and checks them (nothing is written). Mirrored Unity Catalog tables are catalog-managed
Delta, which Spark will not read by path or name, so they are read through the `lh_databricks` SQL analytics endpoint
(`spark.read.synapsesql`); the ERP/WMS shortcuts are read as normal Delta tables. Silver (notebook 03) reads the
same way, so the mirrors stay the single copy of source data.

In [ ]:
import pandas as pd
import com.microsoft.spark.fabric   # registers spark.read.synapsesql
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# Bronze contract: mirrored tables are zero-copy shortcuts (notebook 00), file feeds are physical tables (notebook 01)

DBX_TABLES = ("sell_through_daily", "digital_intent_daily")
SQL_TABLES = ("ref.market", "ref.person", "ref.decision_owner", "ref.synthetic_run", "erp.location",
              "erp.product_style", "erp.product_sku", "erp.price", "erp.vendor", "erp.wholesale_partner",
              "erp.wholesale_order", "erp.wholesale_order_line", "erp.purchase_order", "erp.purchase_order_line",
              "erp.margin_guardrail", "erp.regional_event", "wms.inventory_position", "wms.transfer_lane",
              "wms.inventory_movement")
SQL_OPTIONAL_TABLES = ("erp.allocation_decision", "erp.stock_transfer_order")   # write-back tables, may be empty

FILE_TABLES = {"member_engagement": "DS-11", "returns": "DS-12", "weather_daily": "DS-13", "regional_events": "DS-14"}

BRONZE_TABLES = {
    **{f"dbx_{t}": dict(system="databricks_mirror", source=t, ds="DS-01" if "sell" in t else "DS-02") for t in DBX_TABLES},
    **{t.replace(".", "_"): dict(system="azure_sql_mirror", source=t, ds="DS-03..10,19") for t in SQL_TABLES + SQL_OPTIONAL_TABLES},
    **{f"file_{t}": dict(system="synthetic_generator", source=t, ds=ds) for t, ds in FILE_TABLES.items()},
}
OPTIONAL_BRONZE = {n for n, m in BRONZE_TABLES.items() if m["source"] in SQL_OPTIONAL_TABLES}


def bronze_spark_df(name):
    """One Bronze table with lineage columns added at read time. Mirrored Unity Catalog tables are catalog-managed
    Delta, which Spark will not read by path or name, so they are read through the lh_databricks SQL endpoint."""
    meta = BRONZE_TABLES[name]
    if meta["system"] == "databricks_mirror":
        sdf = spark.read.synapsesql(f"{dbx_sql_source}.{meta['source']}")
        obj = f"appareliq.kinetiq_batch.{meta['source']}"
    else:
        sdf = spark.table(f"{bronze_schema}.{name}")
        obj = meta["source"]
    if "_ingest_ts" not in sdf.columns:
        sdf = (sdf.withColumn("_ingest_ts", F.current_timestamp())
                  .withColumn("_source_system", F.lit(meta["system"]))
                  .withColumn("_source_object", F.lit(obj)))
    return sdf

In [ ]:
results = []
for name, meta in BRONZE_TABLES.items():
    if meta["system"] == "synthetic_generator":
        continue                                    # physical file-feed tables, written by notebook 01
    try:
        sdf = bronze_spark_df(name)
        n = sdf.count()
    except Exception:
        if name in OPTIONAL_BRONZE:
            print(f"optional table {meta['source']} not mirrored yet: skipped")
            continue
        raise
    unlabelled = sdf.filter("record_label IS NULL OR record_label <> 'SYNTHETIC'").count() if "record_label" in sdf.columns else 0
    results.append(dict(table=f"{bronze_schema}.{name}", source=meta["source"], system=meta["system"], ds=meta["ds"],
                        rows=n, unlabelled=unlabelled))
    print(f"{bronze_schema}.{name:<32} {n:>10,}  (shortcut -> {meta['system']}:{meta['source']})")
summary = pd.DataFrame(results)
display(summary)

In [ ]:
# Validation: every required mirrored table is reachable, non-empty and labelled SYNTHETIC
bad = summary[(summary.rows == 0) & ~summary.table.str.split(".").str[1].isin(OPTIONAL_BRONZE)]
assert bad.empty, f"empty mirrored tables: {bad.table.tolist()}"
assert (summary.unlabelled == 0).all(), summary[summary.unlabelled > 0]
missing = [n for n, m in BRONZE_TABLES.items() if m["system"] != "synthetic_generator" and n not in OPTIONAL_BRONZE
           and f"{bronze_schema}.{n}" not in set(summary.table)]
assert not missing, f"missing mirrored tables: {missing}"
print(f"{len(summary)} mirrored tables validated (zero-copy)")